---
## 7. Evaluation — Precision@k / Recall@k
**Work Package: Performance Evaluation**

**Method 1:** Rank one held-out positive against 99 random negatives (repeated
per positive, up to 3 per user)  
**Method 2:** Leave-one-out — hold out the user's most recent interaction

SVD and BPR are compared here with **default hyperparameters** to select the
better algorithm. Hyperparameter tuning of the winning algorithm happens once,
in Section 8 — not here, to avoid tuning twice.


In [ ]:
def prec_at_k(rec, rel, k): return len(set(rec[:k]) & rel) / k if k else 0
def rec_at_k(rec, rel, k):  return len(set(rec[:k]) & rel) / len(rel) if rel else 0
def f1(p, r): return 2*p*r/(p+r) if p+r else 0

K_VALUES = [1, 3, 5, 10]

train_by_user = {str(k):{str(r) for r in v}
                  for k,v in df_train.groupby('user_id')['recipe_id'].apply(set).items()}
test_by_user  = {str(k):{str(r) for r in v}
                  for k,v in df_test.groupby('user_id')['recipe_id'].apply(set).items()}
known_rids = set(R2I.keys())

eval_users = []
eval_test  = {}
for u in test_by_user:
    if u not in train_by_user or u not in U2I: continue
    known_pos = {r for r in test_by_user[u] if r in known_rids}
    if known_pos:
        eval_users.append(u)
        eval_test[u] = known_pos

active_users = [u for u in eval_users if len(train_by_user.get(u,set())) >= 10]
print(f'Eval users: {len(eval_users):,}  |  Active users: {len(active_users):,}')
validate(len(active_users) > 100, 'Sufficient active users', '> 100', f'{len(active_users):,}')


In [ ]:
# ── Reusable evaluation functions (used throughout the rest of the notebook) ──
def eval_method1(score_fn, label, max_users=500):
    prec = {k: [] for k in K_VALUES}
    rec  = {k: [] for k in K_VALUES}
    for uid in active_users[:max_users]:
        seen = train_by_user.get(uid, set())
        all_pos = eval_test[uid]
        for pos in list(all_pos)[:3]:
            neg_pool = list(known_rids - seen - all_pos)
            if len(neg_pool) < 99: continue
            candidates = [pos] + random.sample(neg_pool, 99)
            preds = sorted([(r, score_fn(uid, r)) for r in candidates],
                            key=lambda x: x[1], reverse=True)
            rec_ids = [p[0] for p in preds]
            rel = {pos}
            for k in K_VALUES:
                prec[k].append(prec_at_k(rec_ids, rel, k))
                rec[k].append(rec_at_k(rec_ids, rel, k))
    print(f'{label}:')
    print(f'{"k":<5}{"Precision":>12}{"Recall":>12}')
    for k in K_VALUES:
        print(f'{k:<5}{np.mean(prec[k]):>12.4f}{np.mean(rec[k]):>12.4f}')
    return prec, rec

def eval_method2(score_fn, label, max_users=500):
    prec = {k: [] for k in K_VALUES}
    rec  = {k: [] for k in K_VALUES}
    for uid in active_users[:max_users]:
        seen = train_by_user.get(uid, set())
        pos_list = list(eval_test.get(uid, set()))
        if not pos_list: continue
        held = pos_list[0]
        neg_pool = list(known_rids - seen - {held})
        if len(neg_pool) < 99: continue
        candidates = [held] + random.sample(neg_pool, 99)
        preds = sorted([(r, score_fn(uid, r)) for r in candidates],
                        key=lambda x: x[1], reverse=True)
        rec_ids = [p[0] for p in preds]
        rel = {held}
        for k in K_VALUES:
            prec[k].append(prec_at_k(rec_ids, rel, k))
            rec[k].append(rec_at_k(rec_ids, rel, k))
    print(f'{label}:')
    for k in K_VALUES:
        print(f'  k={k:<3} Precision={np.mean(prec[k]):.4f}  Recall={np.mean(rec[k]):.4f}')
    return prec, rec


In [ ]:
# ── Evaluate SVD and BPR with DEFAULT hyperparameters — algorithm selection ───
print('=== SVD — Method 1 ===')
svd_prec, svd_rec = eval_method1(
    lambda uid,rid: svd.predict(U2I[uid],R2I[rid]).est, 'SVD')

print('\n=== BPR (default) — Method 1 ===')
bpr_prec, bpr_rec = eval_method1(bpr_score, 'BPR (default)')

validate(np.mean(svd_prec[1]) > 0, 'SVD Precision@1 non-zero', '> 0', f'{np.mean(svd_prec[1]):.4f}')
validate(np.mean(bpr_prec[1]) > 0, 'BPR Precision@1 non-zero', '> 0', f'{np.mean(bpr_prec[1]):.4f}')

print('\n=== SVD — Method 2 (LOO) ===')
svd_prec2, svd_rec2 = eval_method2(
    lambda uid,rid: svd.predict(U2I[uid],R2I[rid]).est, 'SVD')

print('\n=== BPR (default) — Method 2 (LOO) ===')
bpr_prec2, bpr_rec2 = eval_method2(bpr_score, 'BPR (default)')


In [ ]:
# ── Select the algorithm to carry forward into Section 8 tuning ───────────────
print('=== Algorithm Selection (default hyperparameters) ===')
print(f'{"k":<5}{"SVD":>10}{"BPR":>10}{"Winner":>10}')
for k in K_VALUES:
    s, b = np.mean(svd_prec[k]), np.mean(bpr_prec[k])
    print(f'{k:<5}{s:>10.4f}{b:>10.4f}{"BPR" if b>s else "SVD":>10}')

ACTIVE_CF_MODEL = 'bpr' if np.mean(bpr_prec[1]) > np.mean(svd_prec[1]) else 'svd'
print(f'\n🏆 Algorithm selected for tuning in Section 8: {ACTIVE_CF_MODEL.upper()}')
print('(Both models evaluated at default settings — the winner gets the full')
print(' Optuna tuning budget next, rather than tuning both and wasting compute)')


In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (m1, m2, title) in zip(axes, [
    ((svd_prec, bpr_prec), None, 'Method 1: Ranking'),
    ((svd_prec2, bpr_prec2), None, 'Method 2: Leave-One-Out')
]):
    svd_d, bpr_d = m1
    ax.plot(K_VALUES, [np.mean(svd_d[k]) for k in K_VALUES],
             marker='o', color=C_PURPLE, linewidth=2, markersize=7, label='SVD')
    ax.plot(K_VALUES, [np.mean(bpr_d[k]) for k in K_VALUES],
             marker='^', color=C_AFTER, linewidth=2, markersize=7, label='BPR')
    ax.plot(K_VALUES, [k/100 for k in K_VALUES],
             marker='s', color=C_FLAG, linestyle='--', markersize=5, label='Random')
    ax.set_xlabel('k'); ax.set_ylabel('Precision@k')
    ax.set_title(title, fontweight='bold'); ax.legend(fontsize=8)

plt.suptitle('SVD vs BPR (default hyperparameters) — Algorithm Selection',
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/svd_vs_bpr.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 8. Hyperparameter Tuning
**Work Package: Hyperparameter Tuning**

Tunes whichever algorithm won in Section 7 (`ACTIVE_CF_MODEL`). This is the
single tuning pass for the project — Section 7 only compared default settings
to pick the algorithm family; this section optimizes that algorithm's
hyperparameters with Optuna.


In [ ]:
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)

TUNE_USERS = active_users[:100]

if ACTIVE_CF_MODEL == 'svd':
    def objective(trial):
        nf  = trial.suggest_int('n_factors',  10, 100)
        reg = trial.suggest_float('reg_all',   0.001,0.5,log=True)
        lr  = trial.suggest_float('lr_all',    0.001,0.05,log=True)
        m   = SVD(n_factors=nf,reg_all=reg,lr_all=lr,n_epochs=20,random_state=42)
        m.fit(trainset)
        hits=[]
        for uid in TUNE_USERS:
            if uid not in U2I: continue
            seen=train_by_user.get(uid,set())
            pos_list=list(eval_test.get(uid,set()))
            if not pos_list: continue
            pos=pos_list[0]
            neg_pool=list(known_rids-seen-eval_test.get(uid,set()))
            if len(neg_pool)<99: continue
            cands=[pos]+random.sample(neg_pool,99)
            preds=sorted([(r,m.predict(U2I[uid],R2I[r]).est) for r in cands],
                          key=lambda x:x[1],reverse=True)
            hits.append(int([p[0] for p in preds].index(pos)+1<=10))
        return np.mean(hits) if hits else 0.0
else:
    def objective(trial):
        kf    = trial.suggest_int('k', 16, 128)
        lr    = trial.suggest_float('learning_rate', 0.001, 0.1, log=True)
        reg   = trial.suggest_float('lambda_reg', 0.0001, 0.05, log=True)
        iters = trial.suggest_int('max_iter', 50, 200)
        m = BPR(k=kf, max_iter=iters, learning_rate=lr,
                 lambda_reg=reg, seed=42, verbose=False)
        m.fit(cornac_data)
        hits=[]
        for uid in TUNE_USERS:
            seen=train_by_user.get(uid,set())
            pos_list=list(eval_test.get(uid,set()))
            if not pos_list: continue
            pos=pos_list[0]
            neg_pool=list(known_rids-seen-eval_test.get(uid,set()))
            if len(neg_pool)<99: continue
            cands=[pos]+random.sample(neg_pool,99)
            try:
                preds=sorted([(r,m.score(cornac_data.uid_map[uid],cornac_data.iid_map[r]))
                              for r in cands if r in cornac_data.iid_map],
                              key=lambda x:x[1],reverse=True)
                hits.append(int([p[0] for p in preds].index(pos)+1<=10))
            except: continue
        return np.mean(hits) if hits else 0.0

print(f'Running Optuna for {ACTIVE_CF_MODEL.upper()} (30 trials)...')
study = optuna.create_study(direction='maximize',
                              sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30)

best = study.best_params
print(f'\nBest Hit@10: {study.best_value:.1%}')
for k,v in best.items(): print(f'  {k:<15}={v}')


In [ ]:
# ── Retrain final model with tuned hyperparameters ─────────────────────────────
if ACTIVE_CF_MODEL == 'svd':
    final_model = SVD(n_factors=best['n_factors'], reg_all=best['reg_all'],
                        lr_all=best['lr_all'], n_epochs=30, random_state=42)
    final_model.fit(trainset)
    svd = final_model  # update global reference used by full_recommend
    with open('models/final_model.pkl','wb') as f:
        pickle.dump({'model':final_model,'type':'svd','params':best},f)
else:
    final_model = BPR(k=best['k'], max_iter=best['max_iter'],
                        learning_rate=best['learning_rate'],
                        lambda_reg=best['lambda_reg'], seed=42)
    final_model.fit(cornac_data)
    bpr = final_model  # update global reference used by full_recommend / bpr_score
    def bpr_score(uid_str, rid_str):
        try:
            return bpr.score(cornac_data.uid_map[uid_str], cornac_data.iid_map[rid_str])
        except KeyError:
            return -999.0
    with open('models/final_model.pkl','wb') as f:
        pickle.dump({'model':final_model,'type':'bpr','params':best},f)

print(f'Final tuned {ACTIVE_CF_MODEL.upper()} model saved.')

# Re-evaluate the tuned model once, for the record
print(f'\n=== Final tuned {ACTIVE_CF_MODEL.upper()} — Method 1 ===')
score_fn = (lambda uid,rid: svd.predict(U2I[uid],R2I[rid]).est) if ACTIVE_CF_MODEL=='svd' else bpr_score
final_prec, final_rec = eval_method1(score_fn, f'{ACTIVE_CF_MODEL.upper()} (tuned)')
